In [ ]:
import os
import subprocess
import time
from clearml import Task, OutputModel
import getpass
from loguru import logger
import json
import pymysql
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import MinMaxScaler
import numpy as np


# 환경 변수 선언 (Configuration)
work_environ = "home"
phase = os.environ.get("PHASE", "dev")
epochs_cnt = os.environ.get("EPOCHS_CNT", 1)
docker_image = os.environ.get("DOCKER_IMAGE", "172.16.11.236:5000/spire/python:3.12-bullseye")
model_type_str = os.environ.get("MODEL_TYPE_STR", "LSTMAe")
process_id = os.environ.get("PROCESS_ID", "P101")
sub_process_id = os.environ.get("SUB_PROCESS_ID", "1002")
project_name = os.environ.get("PROJECT_NAME", "mes")
if phase == "prod":
    db_conf = json.loads(os.environ.get("DB_CONF", '{"host":"172.16.9.60", "port": 3306, "user": "mlops_detect", "password": "QoS908Z1!", "database": "mes_metric"}'))
else:
    if work_environ == "company":
        db_passwd = "QoS908Z1!"
        db_host = "172.16.9.60"
    else:
        db_passwd = "QoS908Z1!"
        db_host = "192.168.0.127"
    db_con_str = '{'+f'"host":"{db_host}", "port": 3306, "user": "mlops_detect", "password": "{db_passwd}", "database": "mes_metric"'+'}'
    db_conf = json.loads(os.environ.get("DB_CONF", db_con_str))
task_name = f"{process_id}_{sub_process_id}"
BUCKET_NAME = "clearml-data"
output_feature_cols = os.environ.get("OUT_FEATURE_COLS", "internal_temp, humidity, rpm").split(', ')
data_begin_days = os.environ.get("DATA_BEGIN_DAYS", -1)
use_clearml = True if os.environ.get("USE_CLEARML", 1) == 1 else False

# ---------------------------------------------------------
# 1. 하이퍼파라미터 설정 (건조기 센서 데이터 최적 Baseline)
# ---------------------------------------------------------
lstmae_params = {
    'seq_len': 60,          # 시퀀스 길이 (예: 1분 단위 수집 시 1시간 분량)
    'n_features': 8,        # 센서 개수 (예: 온도, 습도, 진동)
    'inner_dim': 64,        # LSTM 은닉층 차원
    'bottleneck_dim': 16,   # 압축 차원 (특징 추출 공간)
    'batch_size': 64,       # 배치 크기
    'learning_rate': 0.001, # 초기 학습률
    'dropout': 0.1,         # 시계열 정보 유지를 위한 낮은 드롭아웃
    'epochs': 20            # 학습 에폭 수
}

lstmae_params = json.loads(os.environ.get("PARAMS", json.dumps(lstmae_params)))

if phase != "prod":
    lstmae_params['epochs'] = 2

if work_environ == "home":
    OBJECT_STORAGE_ENDPOINT = 'http://192.168.0.83:9000'  # Garage 서버 주소 (기본 포트 3900)
    AWS_ACCESS_KEY_ID = '7NFFU2I15W8ASBKI4J7T'
    AWS_ACCESS_SECRET_KEY = 'JGl1E8nCeItLJUvrn48y9FzdhZ+nU1+q95NsmxRH'
    AWS_REGION = 'ap-northeast-2'
else:
    OBJECT_STORAGE_ENDPOINT = 'http://172.16.11.235:9000'  # Garage 서버 주소 (기본 포트 3900)
    AWS_ACCESS_KEY_ID = 'QQY84JF5HCFNC814TE75'
    AWS_ACCESS_SECRET_KEY = 'TOzA6qIU0smDLZhQFS7N8jRUVCB+RbdYoyhtJ3Ma'
    AWS_REGION = 'ap-northeast-2'
    # Clearml 정보
    os.environ['CLEARML_WEB_HOST']='http://172.16.8.168:8080'
    os.environ['CLEARML_API_HOST']='http://172.16.8.168:8008'
    os.environ['CLEARML_FILES_HOST']='http://172.16.8.168:8081'
    os.environ['CLEARML_API_ACCESS_KEY']='NA4TVJLF4MNFPAT8JCSYBOVN0QASF2'
    os.environ['CLEARML_API_SECRET_KEY']='g3ur38Bn2GRwzTGDfcEGJy30iPv0Wx43TYDNhN4S4HjVQn5KX6OIpUbCFwTF2uOCQ3c'
    # YOLO의 자동 ClearML 로깅 비활성화
    os.environ['CLEARML_REGISTER_IGNORE'] = 'True'
    os.environ['CLEARML_SKIP_AUTO_STAGED_VCS'] = '1'
    os.environ['CLEARML_VCS_AUTO_CHECKOUT'] = '0'
    os.environ['CLEARML_VCS_IGNORE_EXTENSIONS'] = '1'
    os.environ['CLEARML_SKIP_VCS_AUTO_CONFIGURATION'] = '1'
    os.environ['CLEARML_SKIP_AUTO_STAGED_VCS'] = '1'
    os.environ['CLEARML_SKIP_VCS_AUTO_CONFIGURATION'] = '1'
    os.environ['CLEARML_SKIP_AUTO_STAGED_VCS'] = '1'
    # Git 정보를 찾지 않도록 설정
    os.environ['CLEARML_SKIP_GIT_CHECK'] = '1'
    # 원격지에서 실행 시 Git clone을 시도하지 않음
    os.environ['CLEARML_FORCE_STORE_DIFF'] = '1'

logger.info(f'phase               : {phase}')
logger.info(f'epochs_cnt          : {epochs_cnt}')
logger.info(f'work_environ        : {work_environ}')
logger.info(f'model_type_str      : {model_type_str}')
logger.info(f'process_id          : {process_id}')
logger.info(f'sub_process_id      : {sub_process_id}')
logger.info(f'project_name        : {project_name}')
logger.info(f'task_name           : {task_name}')
logger.info(f'BUCKET_NAME         : {BUCKET_NAME}')
logger.info(f'params              : {json.dumps(lstmae_params)}')
logger.info(f'db_conf             : {db_conf}')
logger.info(f'output_feature_cols : {output_feature_cols}')
logger.info(f'data_begin_days     : {data_begin_days}')


2026-05-18 23:18:21.588 | INFO     | __main__:<module>:93 - phase               : dev
2026-05-18 23:18:21.590 | INFO     | __main__:<module>:94 - epochs_cnt          : 1
2026-05-18 23:18:21.590 | INFO     | __main__:<module>:95 - work_environ        : home
2026-05-18 23:18:21.591 | INFO     | __main__:<module>:96 - model_type_str      : LSTMAe
2026-05-18 23:18:21.592 | INFO     | __main__:<module>:97 - process_id          : P101
2026-05-18 23:18:21.593 | INFO     | __main__:<module>:98 - sub_process_id      : 1002
2026-05-18 23:18:21.595 | INFO     | __main__:<module>:99 - project_name        : mes
2026-05-18 23:18:21.595 | INFO     | __main__:<module>:100 - task_name           : P101_1002
2026-05-18 23:18:21.596 | INFO     | __main__:<module>:101 - BUCKET_NAME         : clearml-data
2026-05-18 23:18:21.596 | INFO     | __main__:<module>:102 - params              : {"seq_len": 60, "n_features": 8, "inner_dim": 64, "bottleneck_dim": 16, "batch_size": 64, "learning_rate": 0.001, "dropout

In [21]:
class IllegalFeatureException(Exception):
    """
    IllegalFeatureException
    """
    def __init__(self, msg):
        super().__init__(msg)

from torch.utils.data import Dataset

class MySQLDataSet(Dataset):
    """
    MySQLDataSet
    """
    def __init__(self, window_size: int, db_conf: dict,
                 process_id: str, sub_process_id: str, output_feature_cols: list,
                 base_scaler=None, limit=-1, from_days=-1):
        self._window_size = window_size
        self._process_id = process_id
        self._sub_process_id = sub_process_id
        self._table_name = f'{process_id}_{sub_process_id}'
        self._database_name = db_conf['database'] if 'database' in db_conf else 'mes'
        self._db_conf = db_conf
        self._db_handle = self.__connect()
        self._offset_pos = 0
        self._direction = "asc"
        self._db_columns = self.__db_get_columns(table_name=self._table_name)
        self._db_columns_str = ', '.join(self._db_columns)
        self._db_columns_str = self._db_columns_str.replace('timestamp', 'UNIX_TIMESTAMP(timestamp)')
        self._limit = limit
        self._current_rpt_cnt = 0
        self._output_feature_cols = output_feature_cols
        self._output_feature_cols_idx = self.__get_features_cols_num()
        if len(self._output_feature_cols_idx) != len(self._output_feature_cols):
            raise IllegalFeatureException(f'Illegal feature. [{self._output_feature_cols}]')
        self._from_days = from_days
        self._scaler = self._init_scaler(base_scaler)

    @property
    def db_columns(self) -> list:
        return self._db_columns

    @property
    def output_feature_cols_idx(self) -> list:
        return self._output_feature_cols_idx

    @property
    def scaler(self):
        return self._scaler

    @property
    def from_days(self):
        return self._from_days

    @from_days.setter
    def from_days(self, val: int):
        self._from_days = val

    def __connect(self):
        return pymysql.connect(**self._db_conf)

    def __get_features_cols_num(self):
        pos = []
        if set(self._output_feature_cols).issubset(set(self._db_columns)):
            for feature in self._output_feature_cols:
                pos.append(self._db_columns.index(feature))
        return pos

    def _init_scaler(self, base_scaler):
        # 전체를 다 읽지 않고 샘플 데이터를 일부만 읽어 스케일러 기준점(Min/Max)을 잡습니다.
        try:
            with self._db_handle.cursor() as cursor:
                data_num = self._get_count()
                logger.info(f'data_num = {data_num}')
                limit = data_num // 10
                logger.info(f'limit = {limit}')
                query = f"SELECT {self._db_columns_str} FROM {self._table_name}"
                if self._from_days > 0:
                    query += f" where timestamp > date_sub(NOW(), INTERVAL {self._from_days} DAY)"
                query += f" LIMIT {limit if limit >= 1 else data_num % 10}"
                logger.info(f'query = {query}')
                cursor.execute(query)
                rows = cursor.fetchall()
                # df = pd.read_sql_query(query, self._db_handle)
                df = pd.DataFrame(rows, columns=self._db_columns)
                if base_scaler is not None:
                    scaler = base_scaler
                    # 기존 데이터 범위를 유지하면서 새 데이터의 최솟값/최댓값을 반영하여 누적 업데이트
                    scaler.partial_fit(df.values)
                else:
                    scaler = MinMaxScaler()
                    scaler.fit(df.values)
                return scaler
        except Exception as ex:
            logger.error(f"Excetion : {ex}")
            raise ex

    def __db_get_columns(self, table_name: str, from_days = -1):
        query = f"""describe {table_name}"""
        columns = []
        try:
            with self._db_handle.cursor() as cursor:
                # DESCRIBE 명령어로 테이블 정보 요청
                cursor.execute(query)
                rows = cursor.fetchall()
                for row in rows:
                    if row[0] not in ("id", "process_id", "sub_process_id"):
                        columns.append(row[0])
        except Exception as ex:
            logger.error(f"Exception: {ex}")
        return columns
    
    def __get_item(self, query: str):
        try:
            with self._db_handle.cursor() as cursor:
                cursor.execute(query)
                rows = cursor.fetchall()
                # logger.info(f'rows = {rows[0]}')
                return pd.DataFrame(rows, columns=self._db_columns)
        except Exception as ex:
            logger.error(f"Exception: {ex}")
        return None

    def _get_count(self, from_days=-1):
        query = f'select count(*) from {self._table_name}'
        if from_days > 0 or self.from_days > 0:
            query += f' where timestamp > date_sub(NOW(), INTERVAL {from_days if from_days > 0 else self._from_days} DAY)'
        logger.info(f'query = {query}')
        count = 0
        try:
            with self._db_handle.cursor() as cursor:
                cursor.execute(query)
                rows = cursor.fetchone()
                count = rows[0]
        except Exception as ex:
            logger.error(f"Exception: {ex}")
        return count

    def __len__(self):
        data_len = self._get_count()
        return data_len - self._window_size

    def __getitem__(self, idx):
        # if self._current_rpt_cnt < self._limit:
        # logger.info(f'idx = {idx}')
        if self._limit > 0 and self._current_rpt_cnt >= self._limit:
            return None
        query = f'SELECT {self._db_columns_str} FROM {self._table_name}'
        if self._from_days > 0:
            query += f" where timestamp > date_sub(NOW(), INTERVAL {self._from_days} DAY)"
        query += f' ORDER BY id {self._direction} LIMIT {self._window_size} OFFSET {idx}'
        # logger.info(f'query = {query}')
        sequence = self.__get_item(query=query)
        self._current_rpt_cnt += 1
        # logger.info(f'sequence = {type(sequence)}, {sequence}')
        # logger.info(f'sequence\'s shape = {sequence.shape}')
        if sequence.empty or len(sequence) < self._window_size:
            # 에러 방지용 기본 텐서 반환시에도 정확히 피처 개수를 8로 맞춥니다.
            return torch.zeros((self._window_size, len(self._db_columns)), dtype=torch.float32)
            # return torch.zeros((self._window_size, 8), dtype=torch.float32)
        # 2. 정규화 및 텐서 변환
        scaled_data = self._scaler.transform(sequence.values)
        return torch.tensor(scaled_data, dtype=torch.float32)
        """
        if df.empty or len(df) < self.seq_len:
                # 에러 방지용 기본 텐서 반환시에도 정확히 피처 개수를 3으로 맞춥니다.
                return torch.zeros((self.seq_len, 3), dtype=torch.float32)
            
            # 2. [핵심] 스케일러 적용 전에 데이터프레임에 우리가 원하는 3개 컬럼만 있는지 재확인
            # 만약 다른 컬럼이 섞여있더라도 아래 코드가 3개 컬럼만 필터링합니다.
            target_features = df[['temperature', 'humidity', 'vibration']].values
            
            # 3. 3차원 데이터만 스케일러에 통과
            scaled_data = self.scaler.transform(target_features)
            
            # 최종 반환 텐서의 크기는 (SEQ_LEN, 3)이 됩니다.
            return torch.tensor(scaled_data, dtype=torch.float32)
        """
    def close(self):
        if self._db_handle is not None:
            self._db_handle.close()
            self._db_handle = None

    def __str__(self):
        return f'MySQLDataSet(table_name=\"{self._table_name}\")'
    

In [22]:
run_this=False
if run_this:
    datasets = MySQLDataSet(window_size=params['seq_len'],
                            db_conf=db_conf,
                            process_id=process_id,
                            sub_process_id=sub_process_id,
                            output_feature_cols=output_feature_cols,
                            limit=2,
                            from_days=-1)
    logger.info(f'columns = {datasets.db_columns}')
    logger.info(f'output_feature_cols_idx = {datasets.output_feature_colsfeature_cols_idx}')
    for data in datasets:
        if data is None or len(data[0]) <= 0:
            break
        logger.info(f'dataset = {data}')
    datasets.close()

In [23]:
"""
STM-AutoEncoder 신경망 모델 정의
"""
import joblib
from sklearn.metrics import r2_score


class Encoder(nn.Module):
    def __init__(self, seq_len, n_features, inner_dim, bottleneck_dim):
        super(Encoder, self).__init__()
        self.lstm1 = nn.LSTM(n_features, inner_dim, batch_first=True, dropout=0.1)
        self.lstm2 = nn.LSTM(inner_dim, bottleneck_dim, batch_first=True)
        
    def forward(self, x):
        x, _ = self.lstm1(x)
        _, (hidden, _) = self.lstm2(x)
        return hidden.squeeze(0)

class Decoder(nn.Module):
    def __init__(self, seq_len, n_features, inner_dim, bottleneck_dim):
        super(Decoder, self).__init__()
        self.seq_len = seq_len
        self.lstm1 = nn.LSTM(bottleneck_dim, inner_dim, batch_first=True, dropout=0.1)
        self.lstm2 = nn.LSTM(inner_dim, n_features, batch_first=True)
        
    def forward(self, x):
        x = x.unsqueeze(1).repeat(1, self.seq_len, 1)
        x, _ = self.lstm1(x)
        x, _ = self.lstm2(x)
        return x

class LSTMAutoEncoder(nn.Module):
    def __init__(self, process_id, sub_process_id,
                 lstmae_params: dict, datasets, device,
                 db_conf=None, pre_scaler_filepath_name: str = None,
                 clearml_task=None):
        super(LSTMAutoEncoder, self).__init__()
        self._process_id = process_id
        self._sub_process_id = sub_process_id
        self._lstmae_params = lstmae_params
        self._db_conf = db_conf
        self._device = device
        self._encoder = Encoder(lstmae_params['seq_len'], lstmae_params['n_features'], lstmae_params['inner_dim'], lstmae_params['bottleneck_dim'])
        self._decoder = Decoder(lstmae_params['seq_len'], lstmae_params['n_features'], lstmae_params['inner_dim'], lstmae_params['bottleneck_dim'])
        self._datasets = datasets
        self._clearml_task = clearml_task
        self._base_scaler = None
        if pre_scaler_filepath_name is not None:
            self._base_scaler = joblib.load(pre_scaler_filepath_name)

    @property
    def datasets(self):
        return self._datasets

    def forward(self, x):
        return self._decoder(self._encoder(x))

    def execute_train(self):
        """
        ClearML 자산 로드 및 점진적 재학습 실행
        """
        logger.info(f"사용중인 연산 디바이스: {self._device}")
        result = False

        self._train_loader = DataLoader(self._datasets, batch_size=self._lstmae_params['batch_size'], shuffle=True, drop_last=True)
        self._criterion = nn.MSELoss()
        self._optimizer = optim.Adam(self.parameters(), lr=self._lstmae_params['learning_rate'])
        self._scheduler = optim.lr_scheduler.ReduceLROnPlateau(self._optimizer, mode='min', patience=2, factor=0.5)
        
        # 3) 모델 트레이닝 루프
        clearml_logger = self._clearml_task.get_logger() if self._clearml_task else None
        self.to(self._device)
        logger.info("--- 훈련 시작 ---")
        self.train()
        try:
            for epoch in range(1, self._lstmae_params['epochs']):
                epoch_loss = 0.0
                logger.info(f'epoch = {epoch} / {self._lstmae_params['epochs']}')
                # 에폭마다 정확도 계산을 위해 원본(정답)과 복원본을 담을 임시 리스트
                all_inputs = []
                all_outputs = []                
                for batch in self._train_loader:
                    # logger.info(f'batch = {len(batch)}')
                    inputs = batch.to(self._device)
                    self._optimizer.zero_grad()
                    outputs = self(inputs)
                    loss = self._criterion(outputs, inputs)
                    loss.backward()
                    self._optimizer.step()
                    epoch_loss += loss.item() * inputs.size(0)
                    # 정확도 연산을 위해 CPU 넘파이 배열로 변환하여 수집
                    all_inputs.append(inputs.detach().cpu().numpy())
                    all_outputs.append(outputs.detach().cpu().numpy())                    
                epoch_loss /= len(self._train_loader.dataset)
                self._scheduler.step(epoch_loss)
                # ---------------------------------------------------------
                # [핵심] 복원 정확도(R² Score) 평탄화 및 메트릭 연산
                # ---------------------------------------------------------
                # 수집된 리스트를 거대한 하나의 넘파이 행렬로 결합
                all_inputs = np.concatenate(all_inputs, axis=0)
                all_outputs = np.concatenate(all_outputs, axis=0)
            
                # r2_score는 2차원 이하의 행렬만 지원하므로, (샘플수 * 시퀀스길이, 피처수)로 평탄화(Flatten)
                flat_inputs = all_inputs.reshape(-1, all_inputs.shape[-1])
                flat_outputs = all_outputs.reshape(-1, all_outputs.shape[-1])
            
                # R² Score 계산 (기본 0~1 사이 값, 완벽 복원시 1.0, 베이스라인 미달시 음수 발생 가능)
                # 퍼센티지 가독성을 위해 100을 곱해 최대 100% 점수로 보정합니다.
                r2_accuracy = r2_score(flat_inputs, flat_outputs) * 100
                # 음수 오차 범위 방어 코드 (학습 초반 모델이 완전히 망가졌을 때 음수 방지)
                r2_accuracy = max(0.0, r2_accuracy)                
                logger.info(f"Epoch [{epoch}/{self._lstmae_params['epochs']}] - Train Loss (MSE): {epoch_loss:.6f}")
                logger.info(f"Epoch [{epoch}/{self._lstmae_params['epochs']}] - Reconstruction Accuracy: {r2_accuracy:.2f}%")
                if clearml_logger:
                    clearml_logger.report_scalar(
                        title="Training_Metrics",  # 대시보드 탭의 대분류 이름
                        series="Loss_MSE",         # 꺾은선 그래프 범례 이름
                        value=epoch_loss,          # Y축 값
                        iteration=epoch            # X축 값 (에폭 번호)
                    )
                    # 2) [추가] 정확도 환산 그래프 플롯
                    clearml_logger.report_scalar(
                        title="Training_Metrics",
                        series="Accuracy_R2_Percentage",
                        value=r2_accuracy,
                        iteration=epoch
                    )
                result = True
            # ---------------------------------------------------------
            # [핵심] 임계치(Threshold) 자동 산출 로직
            # ---------------------------------------------------------
            logger.info("\n--- 최적 임계치(Threshold) 계산 중 ---")
            self.eval() # 평가 모드로 전환하여 오차 측정
            all_losses = []
        
            with torch.no_grad():
                for batch in self._train_loader:
                    inputs = batch.to(self._device)
                    outputs = self(inputs)
                
                    # 각 샘플별 개별 MSE 손실 계산 (Batch 차원만 남기고 평균)
                    # 데이터 형상: (Batch, Seq_len, Features)
                    sample_losses = torch.mean((inputs - outputs) ** 2, dim=(1, 2))
                    all_losses.extend(sample_losses.cpu().numpy())
        
            # 실무 표준 기법: 정상 데이터 오차의 상위 5% 지점(95 분위수)을 임계치로 설정
            # (만약 완벽히 깨끗한 정상 데이터만 있다면 99 분위수나 최대값을 사용해도 좋습니다)
            self.threshold = float(np.percentile(all_losses, 95))
            logger.info(f"산출된 이상 탐지 임계치(95% Percentile): {self.threshold:.6f}")
        
            # ClearML 하이퍼파라미터(Configuration) 영역에 임계값 강제 기록 업데이트
            if self._clearml_task:
                self._clearml_task.connect_configuration({"computed_threshold": self.threshold}, name="Model_Threshold")

        except Exception as ex:
            logger.error(f'Exception : {ex}')
        logger.info(f"--- 훈련 완료 --- [result={result}]")
        return result

    def save_output(self, out_path: str):
        logger.info(f'out_path={out_path}')
        if os.path.isdir(out_path) == False:
            os.mkdir(out_path)
        weight_filepath_name = f'{out_path}/lstm_ae_weights.pth'
        scaler_filepath_name = f'{out_path}/minmax_scaler.pkl'        
        torch.save(self.state_dict(), weight_filepath_name)
        joblib.dump(self._datasets.scaler, scaler_filepath_name)
        logger.info("가중치('lstm_ae_weights.pth') 및 스케일러('minmax_scaler.pkl') 파일 추출 완료!")
        weight_filepath_name, scaler_filepath_name
        return weight_filepath_name, scaler_filepath_name


In [24]:
# 이전 모델 정보 다운 로드

def download_artifacts_from_clearml(task_id):
    """
    ClearML 서버에서 특정 프로젝트 및 태스크 이름에 맵핑된 
    가중치와 스케일러 파일을 찾아 로컬에 다운로드합니다.
    """
    task = Task.get_task(task_id=task_id)
    
    if task is None:
        raise ValueError(f"지정한 프로젝트 또는 태스크를 ClearML에서 찾을 수 없습니다.")

    # 2) Task에 등록된 artifacts 딕셔너리에서 파일들을 가져옵니다.
    artifacts = task.artifacts
    
    if "lstm_autoencoder_weights" not in artifacts or "minmax_scaler" not in artifacts:
        logger.info("태스크 내에 'lstm_autoencoder_weights' 또는 'minmax_scaler' Artifact가 존재하지 않습니다.")
        return None, None

    # 3) get_local_copy() 함수를 호출하면 ClearML 원격 스토리지(S3 등)에서 
    #    현재 로컬 머신의 임시 캐시 디렉토리로 파일을 자동 다운로드하고 그 절대경로를 반환합니다.
    local_weight_path = artifacts["lstm_autoencoder_weights"].get_local_copy()
    local_scaler_path = artifacts["minmax_scaler"].get_local_copy()
    
    logger.info("ClearML로부터 다운로드 성공!")
    logger.info(f"-> 가중치 로컬 경로: {local_weight_path}")
    logger.info(f"-> 스케일러 로컬 경로: {local_scaler_path}")
    
    return local_weight_path, local_scaler_path


def get_pretrained_weigth(project_name, task_name):
    """
    마지막으로 훈련에 성공한 모델의 가중치 파일 다운 받습니다.
    """
    tasks = Task.query_tasks(
        project_name=project_name,
        task_name=task_name,
        task_filter={
            'status': ['completed'],
            'order_by': ['-created']  # Ensures the most recently updated is first
        }
    )
    logger.info(f'tasks\'len = {len(tasks)}')
    if len(tasks) > 0:
        last_task_id= tasks[0]
        logger.info(f'last_task_id = {last_task_id}')
        local_weight_path, local_scaler_path = download_artifacts_from_clearml(task_id=last_task_id)
    logger.info(f"Weight downloaded to #3: {local_weight_path}")
    logger.info(f"Scaler downloaded to #3: {local_scaler_path}")
    return local_weight_path, local_scaler_path


In [25]:
def train_and_extract_weights(lstmae_params: dict,
                              pre_weight_filepath_name: str=None,
                              pre_scaler_filepath_name: str=None,
                              clearml_task=None):
    """
    ClearML 자산 로드 및 점진적 재학습 실행    
    """
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    logger.info(f"사용중인 연산 디바이스: {device}")

    base_scaler = None
    if pre_scaler_filepath_name is not None:
        base_scaler = joblib.load(pre_scaler_filepath_name)
    
    # 1) 데이터셋 및 데이터로더 초기화
    datasets = MySQLDataSet(window_size=lstmae_params['seq_len'],
                            db_conf=db_conf,
                            process_id=process_id,
                            sub_process_id=sub_process_id,
                            output_feature_cols=output_feature_cols,
                            base_scaler=base_scaler,
                            limit=-1)
    model = LSTMAutoEncoder( process_id=process_id, sub_process_id=sub_process_id,
                             lstmae_params=lstmae_params, datasets=datasets, device=device,
                             db_conf=db_conf, pre_scaler_filepath_name=pre_scaler_filepath_name,
                             clearml_task=clearml_task)
    # if pre_weight_filepath_name is not None:
    #     logger.info(f'pre_weight_filepath_name = {pre_weight_filepath_name}')
    #     model.load_state_dict(torch.load(pre_weight_filepath_name, map_location=device))
    result = model.execute_train()
    datasets.close()
    logger.info(f'train result = {result}')
    if result:
        weight_filepath_name, scaler_filepath_name = model.save_output(out_path='weights')
    else:
        weight_filepath_name = scaler_filepath_name = None
    return weight_filepath_name, scaler_filepath_name


In [26]:
logger.info(f'project_name = {project_name}')
logger.info(f'task_name    = {task_name}')
task = None
use_clearml = False
if use_clearml:
    task = Task.init(project_name=project_name,
                     task_name=task_name,
                     reuse_last_task_id=False,
                     continue_last_task=False,
                     task_type=Task.TaskTypes.training)
    if len(lstmae_params.keys()) > 0:
        task.connect(lstmae_params)
    task.set_parameter('version', '1.0')
    task.set_task_type('training')
    ia_task_initiated = False

    logger.info(f'run in phase = {phase}')
    # task.set_base_docker(docker_image)
    # task.set_container(
    #     # 원격 에이전트가 기반으로 사용할 기본 도커 이미지 (필수 입력)
    #     image=docker_image, 
    #     # 여기에 도커 실행 옵션들을 공백으로 구분한 하나의 문자열로 전달합니다.
    #     arguments="--privileged --device /dev/fuse --cap-add SYS_ADMIN"
    # )

    task.set_base_docker(
        docker_image=docker_image,  # 원격에서 실행할 베이스 이미지
        docker_arguments="--privileged --device /dev/fuse"  # 도커 실행 인자
    )

    if phase == "prod":
        task.execute_remotely(queue_name='services',
                              clone=True,
                              exit_process=False)
        from clearml.config import running_remotely

        if not running_remotely():
            # [로컬 Parent 프로세스 영역]
            print("Parent: 자식 태스크를 원격 에이전트에 전달했습니다.")
            task.close()
            task.delete(delete_artifacts_and_models=True)
            exit()
if use_clearml:
    pre_weight_filepath_name, pre_scaler_filepath_name = get_pretrained_weigth(project_name=project_name,
                                                                               task_name=task_name)
else:
    pre_weight_filepath_name = None
    pre_scaler_filepath_name = None
logger.info(f'pre_weight_filepath_name = {pre_weight_filepath_name}')
logger.info(f'pre_scaler_filepath_name = {pre_scaler_filepath_name}')
weight_filepath_name, scaler_filepath_name = train_and_extract_weights(lstmae_params=lstmae_params,
                                                                       pre_weight_filepath_name=pre_weight_filepath_name,
                                                                       pre_scaler_filepath_name=pre_scaler_filepath_name)

2026-05-18 23:18:25.592 | INFO     | __main__:<module>:1 - project_name = mes
2026-05-18 23:18:25.594 | INFO     | __main__:<module>:2 - task_name    = P101_1002
2026-05-18 23:18:25.597 | INFO     | __main__:<module>:49 - pre_weight_filepath_name = None
2026-05-18 23:18:25.598 | INFO     | __main__:<module>:50 - pre_scaler_filepath_name = None
2026-05-18 23:18:25.599 | INFO     | __main__:train_and_extract_weights:9 - 사용중인 연산 디바이스: cpu


OperationalError: (1045, "Access denied for user 'mlops_detect'@'DESKTOP-E6CQACF' (using password: YES)")

In [27]:
## 가중치 파일과 스케일 파일 저장 및 종료
if use_clearml:
    try:
        task.upload_artifact(name="lstm_autoencoder_weights", artifact_object=weight_filepath_name)
        task.upload_artifact(name="minmax_scaler", artifact_object=scaler_filepath_name)
        logger.info(f"모델이 파일로 저장되었습니다. [{weight_filepath_name}]")
        os.remove(weight_filepath_name)
        os.remove(scaler_filepath_name)
    except Exception as ex:
        logger.info(f'Exception : {ex}')
    finally:
        task.close()